In [1]:
import numpy as np   
import pandas as pd

In [2]:
from sklearn.preprocessing import LabelEncoder


In [4]:
df=pd.read_csv('cleaned_lalpurja_house_after_eda.csv')

In [5]:
df_lph=df.copy()

In [6]:
df_lph.sample(10)

,district,property_type,property_face,road_type,furnishing,municipality,ward_no,neighborhood,total_price,land_size_aana,...,hospital_m,airport_m,pharmacy_m,bhatbhateni_m,school_m,college_m,public_transport_m,police_station_m,boudhanath_m,ring_road_m
616,Kathmandu,Residential,East,Low Access,Semi Furnished,Budhanilkantha,3.0,Budhanilkantha,70000000.0,9.00,...,800.0,10000.0,500.0,4000.0,200.0,500.0,500.0,500.0,8000.0,3000.0
1904,Lalitpur,Residential,South-East,High Access,Semi Furnished,Lalitpur,4.0,Hattiban,135000000.0,14.50,...,200.0,5000.0,50.0,800.0,50.0,100.0,60.0,100.0,13000.0,300.0
1293,Kathmandu,Residential,East,Low Access,Unfurnished,Kirtipur,2.0,Kirtipur,18500000.0,4.00,...,880.0,13000.0,200.0,2040.0,230.0,590.0,202.0,440.0,13300.0,5900.0
1810,Lalitpur,Residential,North-East,High Access,Unfurnished,Lalitpur,23.0,Dhapakhel,36200000.0,6.00,...,500.0,9000.0,240.0,3000.0,500.0,270.0,100.0,1500.0,14000.0,2000.0
144,Kathmandu,Residential,West,High Access,Unfurnished,Kathmandu,32.0,Pepsicola,29500000.0,3.87,...,2000.0,4500.0,300.0,2000.0,300.0,2000.0,100.0,500.0,6000.0,3300.0
1953,Lalitpur,Residential,South,High Access,Unfurnished,Lalitpur,25.0,Bhaisepati,67500000.0,6.00,...,780.0,6300.0,161.0,2040.0,181.0,221.0,153.0,400.0,3000.0,1704.0
1725,Lalitpur,Residential,East,High Access,Semi Furnished,Mahalaxmi,8.0,Lubhu,22000000.0,3.25,...,400.0,5000.0,100.0,300.0,100.0,100.0,100.0,200.0,12000.0,5000.0
2020,Bhaktapur,Residential,South,High Access,Semi Furnished,Suryabinayak,5.0,Radhe Radhe,39000000.0,3.87,...,1500.0,9000.0,700.0,800.0,1000.0,1000.0,800.0,800.0,8500.0,12000.0
1340,Kathmandu,Residential,North-West,High Access,Unfurnished,Gokarneshwor,6.0,Jorpati,39000000.0,6.20,...,400.0,6100.0,400.0,2400.0,400.0,1100.0,200.0,400.0,2000.0,3500.0
2166,Bhaktapur,Residential,East,Low Access,Semi Furnished,Suryabinayak,5.0,Katunje,16800000.0,4.00,...,1500.0,5000.0,300.0,2000.0,120.0,1000.0,500.0,1000.0,7000.0,6000.0


In [3]:
# ── ENCODING FIRST ─────────────────────────────────────────────────────────


In [7]:
# Label encode low cardinality categoricals
# Why label encode: These have 2-8 unique values
# simple numeric mapping is sufficient
le = LabelEncoder()
for col in ['district', 'road_type', 'property_type',
            'property_face', 'furnishing']:
    df_lph[col] = le.fit_transform(df_lph[col])
    print(f"✅ Label encoded: {col}")

✅ Label encoded: district
✅ Label encoded: road_type
✅ Label encoded: property_type
✅ Label encoded: property_face
✅ Label encoded: furnishing


In [8]:
# Target encode high cardinality categoricals
# Why target encode: neighborhood has 142 unique values
# label encoding would give arbitrary numbers
# target encoding replaces with mean log(price) of that category
# so model learns price signal directly from the encoding
log_price = np.log1p(df_lph['total_price'])

for col in ['neighborhood', 'municipality']:
    target_mean = df_lph.groupby(col)[log_price.name]\
        .transform('mean') if log_price.name in df_lph.columns \
        else df_lph.groupby(col)['total_price']\
        .transform(lambda x: np.log1p(x).mean())
    df_lph[col + '_encoded'] = target_mean
    print(f"✅ Target encoded: {col}")

✅ Target encoded: neighborhood
✅ Target encoded: municipality


In [9]:
# ── ENGINEERED FEATURES ────────────────────────────────────────────────────

# log_land — normalize heavily skewed land size
# Why: land_size_aana skewness=5.177 — log reduces this
# Makes model treat size differences proportionally
# Example: diff between 1→2 aana same weight as 4→8 aana
df_lph['log_land'] = np.log1p(df_lph['land_size_aana'])
print(f"✅ log_land created")

✅ log_land created


In [10]:
# log_built — normalize skewed built up area
# Why: built_up_sqft skewness=2.817 — log reduces this
# Example: diff between 500→1000 sqft same weight as 2000→4000 sqft
df_lph['log_built'] = np.log1p(df_lph['built_up_sqft'])
print(f"✅ log_built created")

✅ log_built created


In [11]:
# floor_area_ratio — built up area relative to land size
# Why: FAR captures how densely built the plot is
# High FAR = multi-storey building on small plot = urban
# Low FAR = large plot with small house = suburban/rural
# Example: 1369 sqft on 4 aana (1369sqft) → FAR=1.0 (single floor)
#          2738 sqft on 4 aana → FAR=2.0 (double floor)
df_lph['floor_area_ratio'] = df_lph['built_up_sqft'] / \
    (df_lph['land_size_aana'] * 342.25)
print(f"✅ floor_area_ratio created")

✅ floor_area_ratio created


In [12]:
# urban_centrality — combined airport and ring road proximity
# Why: Both capture urbanness — same as lalpurja land
# Closer to both = more urban = higher price
# Example: Central Kathmandu → high score
#          Outskirt Bhaktapur → low score
df_lph['urban_centrality'] = 1 / \
    (np.log1p(df_lph['airport_m']) + np.log1p(df_lph['ring_road_m']))
print(f"✅ urban_centrality created")

✅ urban_centrality created


In [13]:
# amenity_access_score — composite amenity access
# Why: Single score less noisy than individual distances
# Higher = better access to all amenities = higher price
# Example: All amenities within 500m → score ~0.9
df_lph['amenity_access_score'] = (
    1 / np.log1p(df_lph['hospital_m']) +
    1 / np.log1p(df_lph['school_m'] + 1) +
    1 / np.log1p(df_lph['pharmacy_m'] + 1) +
    1 / np.log1p(df_lph['public_transport_m'] + 1)
)
print(f"✅ amenity_access_score created")

✅ amenity_access_score created


In [14]:
# house_size_score — land × built up interaction
# Why: Big plot AND big house together = most valuable
# Neither alone tells the full story
# Example: 8 aana + 3000 sqft → log(9)×log(3001) = 17.9
#          3 aana + 800 sqft  → log(4)×log(801)  = 9.5
df_lph['house_size_score'] = np.log1p(df_lph['land_size_aana']) * \
    np.log1p(df_lph['built_up_sqft'])
print(f"✅ house_size_score created")

✅ house_size_score created


In [15]:
# comm_road_premium — commercial × road width interaction
# Why: Wide road essential for commercial properties
# Same as lalpurja land — captures joint effect
# Example: Commercial + 20ft road → premium signal
#          Residential + 20ft → no premium
df_lph['comm_road_premium'] = df_lph['property_type'] * \
    df_lph['road_width_feet']
print(f"✅ comm_road_premium created")

✅ comm_road_premium created


In [16]:
# is_corner_plot — diagonal facing = corner plot
# Why: Corner plots have two-side road access = premium
# Same as lalpurja land dataset
df_lph['is_corner_plot'] = df_lph['property_face'].apply(
    lambda x: 1 if x in [
        le.transform(['North-East'])[0] if 'North-East' in le.classes_ else -1,
        le.transform(['North-West'])[0] if 'North-West' in le.classes_ else -1,
        le.transform(['South-East'])[0] if 'South-East' in le.classes_ else -1,
        le.transform(['South-West'])[0] if 'South-West' in le.classes_ else -1
    ] else 0
)
print(f"✅ is_corner_plot created: {df_lph['is_corner_plot'].sum()} corner plots")

✅ is_corner_plot created: 0 corner plots


In [18]:
df_lph['property_face'].value_counts()

property_face
0    528
4    332
7    314
5    283
2    225
1    219
6    165
3    124
Name: count, dtype: int64

In [19]:
# neighborhood_x_district — interaction feature
# Why: Same neighborhood means different things in different districts
# Example: "Imadol" in Lalitpur ≠ "Imadol" in Kathmandu
df_lph['neighborhood_x_district'] = \
    df_lph['neighborhood_encoded'] * df_lph['district']
print(f"✅ neighborhood_x_district created")

✅ neighborhood_x_district created


In [20]:
# municipality_x_ward — interaction feature
# Why: Ward 6 in Kathmandu ≠ Ward 6 in Bhaktapur
# Captures ward-level granularity within each municipality
df_lph['municipality_x_ward'] = \
    df_lph['municipality_encoded'] * df_lph['ward_no']
print(f"✅ municipality_x_ward created")

✅ municipality_x_ward created


In [21]:
# age_condition_score — house age × furnishing interaction
# Why: New fully furnished house commands highest premium
# Old unfurnished house commands lowest
# Example: age=1, Full Furnished → 1×2=2 (high score)
#          age=20, Unfurnished   → 20×0=0 (low score)
# furnishing encoded: check values
print(f"\nFurnishing encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")
df_lph['age_condition_score'] = (1 / (df_lph['house_age'] + 1)) * \
    (df_lph['furnishing'] + 1)
print(f"✅ age_condition_score created")


Furnishing encoding: {'Full Furnished': np.int64(0), 'Semi Furnished': np.int64(1), 'Unfurnished': np.int64(2)}
✅ age_condition_score created


In [22]:
# ── FINAL CHECK ─────────────────────────────────────────────────────────────
print(f"\n=== Feature Engineering Summary ===")
print(f"Features before: 23")
print(f"Features after:  {df_lph.shape[1]}")
print(f"New features: {df_lph.shape[1] - 23}")
print(f"Nulls: {df_lph.isnull().sum().sum()}")

new_features = ['log_land', 'log_built', 'floor_area_ratio',
                'urban_centrality', 'amenity_access_score',
                'house_size_score', 'comm_road_premium',
                'is_corner_plot', 'neighborhood_x_district',
                'municipality_x_ward', 'age_condition_score',
                'neighborhood_encoded', 'municipality_encoded']

print(f"\nNew feature ranges:")
for col in new_features:
    print(f"  {col}: min={df_lph[col].min():.3f}, max={df_lph[col].max():.3f}")


=== Feature Engineering Summary ===
Features before: 23
Features after:  36
New features: 13
Nulls: 0

New feature ranges:
  log_land: min=0.693, max=3.932
  log_built: min=4.789, max=9.210
  floor_area_ratio: min=0.080, max=7.012
  urban_centrality: min=0.050, max=0.217
  amenity_access_score: min=0.542, max=inf
  house_size_score: min=3.757, max=31.158
  comm_road_premium: min=0.000, max=100.000
  is_corner_plot: min=0.000, max=0.000
  neighborhood_x_district: min=0.000, max=253071428.571
  municipality_x_ward: min=20920000.000, max=1948264705.882
  age_condition_score: min=0.019, max=3.000
  neighborhood_encoded: min=16166666.667, max=126535714.286
  municipality_encoded: min=20920000.000, max=55664705.882


In [23]:
# ─────────────────────────────────────────
# RESET AND REDO FEATURE ENGINEERING
# Fix all 4 issues found
# ─────────────────────────────────────────

# First reload clean EDA file to start fresh
df_lph = pd.read_csv('cleaned_lalpurja_house_after_eda.csv')
print(f"Reloaded clean file: {df_lph.shape}")

# ── FIX 1 — CLIP DISTANCE FLOORS TO MINIMUM 50m ───────────────────────────
# Why: 0m distances cause division by zero in amenity_access_score
# log1p(0) = 0 → 1/0 = infinity
# Clipping to 50m minimum prevents this
dist_cols = ['hospital_m', 'airport_m', 'pharmacy_m', 'bhatbhateni_m',
             'school_m', 'college_m', 'public_transport_m',
             'police_station_m', 'boudhanath_m', 'ring_road_m']
for col in dist_cols:
    df_lph[col] = df_lph[col].clip(lower=50)
    zeros = (df_lph[col] == 0).sum()
print(f"✅ All distance columns clipped to minimum 50m")
print(f"   Zeros remaining: {sum((df_lph[c] == 0).sum() for c in dist_cols)}")

# ── ENCODING ───────────────────────────────────────────────────────────────

# Store facing values BEFORE label encoding
# Why: We need original string values to detect diagonal facings
diagonal_facings = ['North-East', 'North-West', 'South-East', 'South-West']
df_lph['is_corner_plot'] = df_lph['property_face']\
    .isin(diagonal_facings).astype(int)
print(f"\n✅ is_corner_plot created BEFORE encoding")
print(f"   Corner plots: {df_lph['is_corner_plot'].sum()}")

# Label encode low cardinality categoricals
le = LabelEncoder()
for col in ['district', 'road_type', 'property_type',
            'property_face', 'furnishing']:
    df_lph[col] = le.fit_transform(df_lph[col])
    print(f"✅ Label encoded: {col}")

# ── FIX 2 — TARGET ENCODE WITH LOG PRICE ──────────────────────────────────
# Why log price: Raw price in millions creates huge encoded values
# log1p(35M) ≈ 17.4 → much more manageable scale
# Model learns relative price differences not absolute values
log_price = np.log1p(df_lph['total_price'])

for col in ['neighborhood', 'municipality']:
    df_lph[col + '_encoded'] = df_lph.groupby(col)['total_price']\
        .transform(lambda x: np.log1p(x).mean())
    print(f"✅ Target encoded with LOG price: {col}")

# ── ENGINEERED FEATURES ────────────────────────────────────────────────────

# log transforms for skewed features
df_lph['log_land']  = np.log1p(df_lph['land_size_aana'])
df_lph['log_built'] = np.log1p(df_lph['built_up_sqft'])
print(f"\n✅ log_land created")
print(f"✅ log_built created")

# floor_area_ratio — built up relative to land size
# High FAR = multi-storey urban building
# Low FAR = large plot with small house = suburban
df_lph['floor_area_ratio'] = df_lph['built_up_sqft'] / \
    (df_lph['land_size_aana'] * 342.25)
print(f"✅ floor_area_ratio created")

# urban_centrality — airport + ring road proximity combined
df_lph['urban_centrality'] = 1 / \
    (np.log1p(df_lph['airport_m']) + np.log1p(df_lph['ring_road_m']))
print(f"✅ urban_centrality created")

# ── FIX 3 — amenity_access_score with clipped distances ───────────────────
# All distances now minimum 50m so log1p(50)=3.93 → no division by zero
df_lph['amenity_access_score'] = (
    1 / np.log1p(df_lph['hospital_m']) +
    1 / np.log1p(df_lph['school_m']) +
    1 / np.log1p(df_lph['pharmacy_m']) +
    1 / np.log1p(df_lph['public_transport_m'])
)
print(f"✅ amenity_access_score created (no inf)")

# house_size_score — land × built up interaction
df_lph['house_size_score'] = np.log1p(df_lph['land_size_aana']) * \
    np.log1p(df_lph['built_up_sqft'])
print(f"✅ house_size_score created")

# comm_road_premium — commercial × road width
df_lph['comm_road_premium'] = df_lph['property_type'] * \
    df_lph['road_width_feet']
print(f"✅ comm_road_premium created")

# neighborhood_x_district — same neighborhood in different districts
df_lph['neighborhood_x_district'] = \
    df_lph['neighborhood_encoded'] * df_lph['district']
print(f"✅ neighborhood_x_district created")

# municipality_x_ward — ward level granularity within municipality
df_lph['municipality_x_ward'] = \
    df_lph['municipality_encoded'] * df_lph['ward_no']
print(f"✅ municipality_x_ward created")

# age_condition_score — new + furnished = premium
# 1/(age+1) gives higher score to newer houses
# multiplied by furnishing level
df_lph['age_condition_score'] = (1 / (df_lph['house_age'] + 1)) * \
    (df_lph['furnishing'] + 1)
print(f"✅ age_condition_score created")

# ring_road_proximity — reciprocal log
# Same as lalpurja land — captures exponential value decay
df_lph['ring_road_proximity'] = 1 / np.log1p(df_lph['ring_road_m'])
print(f"✅ ring_road_proximity created")

# ── FINAL CHECK ─────────────────────────────────────────────────────────────
new_features = ['log_land', 'log_built', 'floor_area_ratio',
                'urban_centrality', 'amenity_access_score',
                'house_size_score', 'comm_road_premium', 'is_corner_plot',
                'neighborhood_x_district', 'municipality_x_ward',
                'age_condition_score', 'ring_road_proximity',
                'neighborhood_encoded', 'municipality_encoded']

print(f"\n=== Feature Engineering Summary ===")
print(f"Features before: 23")
print(f"Features after:  {df_lph.shape[1]}")
print(f"Nulls: {df_lph.isnull().sum().sum()}")
print(f"Inf values: {np.isinf(df_lph.select_dtypes(include=np.number)).sum().sum()}")

print(f"\nNew feature ranges:")
for col in new_features:
    print(f"  {col}: min={df_lph[col].min():.3f}, max={df_lph[col].max():.3f}")

Reloaded clean file: (2190, 23)
✅ All distance columns clipped to minimum 50m
   Zeros remaining: 0

✅ is_corner_plot created BEFORE encoding
   Corner plots: 797
✅ Label encoded: district
✅ Label encoded: road_type
✅ Label encoded: property_type
✅ Label encoded: property_face
✅ Label encoded: furnishing
✅ Target encoded with LOG price: neighborhood
✅ Target encoded with LOG price: municipality

✅ log_land created
✅ log_built created
✅ floor_area_ratio created
✅ urban_centrality created
✅ amenity_access_score created (no inf)
✅ house_size_score created
✅ comm_road_premium created
✅ neighborhood_x_district created
✅ municipality_x_ward created
✅ age_condition_score created
✅ ring_road_proximity created

=== Feature Engineering Summary ===
Features before: 23
Features after:  37
Nulls: 0
Inf values: 0

New feature ranges:
  log_land: min=0.693, max=3.932
  log_built: min=4.789, max=9.210
  floor_area_ratio: min=0.080, max=7.012
  urban_centrality: min=0.050, max=0.117
  amenity_access_sc

In [24]:
# ─────────────────────────────────────────
# SAVE FEATURE ENGINEERED LALPURJA HOUSING DATASET
# Final version ready for modeling with:
# - 23 original cleaned features
# - 14 engineered features
# - 37 total features, 2190 rows, zero nulls
# ─────────────────────────────────────────
df_lph.to_csv('lalpurja_house_features_ready.csv', index=False)

print(f"✅ Saved lalpurja_house_features_ready.csv")
print(f"✅ Shape: {df_lph.shape}")
print(f"✅ Nulls: {df_lph.isnull().sum().sum()}")
print(f"✅ Columns: {list(df_lph.columns)}")

✅ Saved lalpurja_house_features_ready.csv
✅ Shape: (2190, 37)
✅ Nulls: 0
✅ Columns: ['district', 'property_type', 'property_face', 'road_type', 'furnishing', 'municipality', 'ward_no', 'neighborhood', 'total_price', 'land_size_aana', 'built_up_sqft', 'house_age', 'road_width_feet', 'hospital_m', 'airport_m', 'pharmacy_m', 'bhatbhateni_m', 'school_m', 'college_m', 'public_transport_m', 'police_station_m', 'boudhanath_m', 'ring_road_m', 'is_corner_plot', 'neighborhood_encoded', 'municipality_encoded', 'log_land', 'log_built', 'floor_area_ratio', 'urban_centrality', 'amenity_access_score', 'house_size_score', 'comm_road_premium', 'neighborhood_x_district', 'municipality_x_ward', 'age_condition_score', 'ring_road_proximity']
